In [ ]:
import csv
import os
import heapq
import tempfile
import random

In [ ]:
employee_file = "employees.csv"

with open(employee_file, "w", newline="") as file:

    writer = csv.writer(file)

    writer.writerow([
        "EmployeeID",
        "Name",
        "DepartmentID"
    ])

    for i in range(1, 10001):

        writer.writerow([
            i,
            f"Employee_{i}",
            random.randint(1, 100)
        ])

print("Employee dataset created successfully.")

Employee dataset created successfully.


In [ ]:
department_file = "departments.csv"

with open(department_file, "w", newline="") as file:

    writer = csv.writer(file)

    writer.writerow([
        "DepartmentID",
        "DepartmentName"
    ])

    for i in range(1, 101):

        writer.writerow([
            i,
            f"Department_{i}"
        ])

print("Department dataset created successfully.")

Department dataset created successfully.


In [ ]:
def create_sorted_chunks(input_file, key_index, chunk_size=1000):

    chunk_files = []

    with open(input_file, "r", newline="") as file:

        reader = csv.reader(file)

        # Read header
        header = next(reader)

        chunk = []

        for row in reader:

            chunk.append(row)

            # When chunk reaches memory limit
            if len(chunk) >= chunk_size:

                # Sort chunk
                chunk.sort(
                    key=lambda x: int(x[key_index])
                )

                # Create temporary disk file
                temp_file = tempfile.NamedTemporaryFile(
                    mode="w",
                    delete=False,
                    newline="",
                    suffix=".csv"
                )

                writer = csv.writer(temp_file)

                # Write sorted chunk
                for record in chunk:
                    writer.writerow(record)

                temp_file.close()

                # Store temporary file name
                chunk_files.append(
                    temp_file.name
                )

                # Clear memory
                chunk = []

        # Process remaining rows
        if chunk:

            chunk.sort(
                key=lambda x: int(x[key_index])
            )

            temp_file = tempfile.NamedTemporaryFile(
                mode="w",
                delete=False,
                newline="",
                suffix=".csv"
            )

            writer = csv.writer(temp_file)

            for record in chunk:
                writer.writerow(record)

            temp_file.close()

            chunk_files.append(
                temp_file.name
            )

    return header, chunk_files

In [ ]:
employee_header, employee_chunks = create_sorted_chunks(
    employee_file,
    key_index=2,
    chunk_size=1000
)

print("Number of Employee chunks:", len(employee_chunks))

Number of Employee chunks: 10


In [ ]:
department_header, department_chunks = create_sorted_chunks(
    department_file,
    key_index=0,
    chunk_size=20
)

print("Number of Department chunks:", len(department_chunks))

Number of Department chunks: 5


In [ ]:
def merge_sorted_chunks(
    chunk_files,
    output_file,
    key_index,
    header
):

    files = []
    readers = []

    # Open all sorted chunk files
    for filename in chunk_files:

        file = open(
            filename,
            "r",
            newline=""
        )

        files.append(file)

        readers.append(
            csv.reader(file)
        )

    heap = []

    # Read first row from every chunk
    for index, reader in enumerate(readers):

        try:

            row = next(reader)

            key = int(
                row[key_index]
            )

            heapq.heappush(
                heap,
                (
                    key,
                    index,
                    row
                )
            )

        except StopIteration:
            pass

    # Write final sorted output
    with open(
        output_file,
        "w",
        newline=""
    ) as output:

        writer = csv.writer(output)

        # Write header
        writer.writerow(header)

        while heap:

            key, file_index, row = heapq.heappop(
                heap
            )

            # Write smallest row
            writer.writerow(row)

            try:

                # Read next row from same file
                next_row = next(
                    readers[file_index]
                )

                next_key = int(
                    next_row[key_index]
                )

                heapq.heappush(
                    heap,
                    (
                        next_key,
                        file_index,
                        next_row
                    )
                )

            except StopIteration:
                pass

    # Close all files
    for file in files:
        file.close()

In [ ]:
sorted_employee_file = "sorted_employees.csv"

merge_sorted_chunks(
    employee_chunks,
    sorted_employee_file,
    key_index=2,
    header=employee_header
)

print("Employee table sorted successfully.")

Employee table sorted successfully.


In [ ]:
sorted_department_file = "sorted_departments.csv"

merge_sorted_chunks(
    department_chunks,
    sorted_department_file,
    key_index=0,
    header=department_header
)

print("Department table sorted successfully.")

Department table sorted successfully.


In [ ]:
def streaming_merge_join(
    employee_file,
    department_file,
    output_file
):

    with open(
        employee_file,
        "r",
        newline=""
    ) as emp_file, open(
        department_file,
        "r",
        newline=""
    ) as dept_file, open(
        output_file,
        "w",
        newline=""
    ) as output:

        emp_reader = csv.reader(emp_file)
        dept_reader = csv.reader(dept_file)

        writer = csv.writer(output)

        # Read headers
        emp_header = next(emp_reader)
        dept_header = next(dept_reader)

        # Output header
        writer.writerow([
            "EmployeeID",
            "Name",
            "DepartmentID",
            "DepartmentName"
        ])

        try:
            emp_row = next(emp_reader)
            dept_row = next(dept_reader)

        except StopIteration:
            return

        # Streaming Merge Join
        while True:

            emp_key = int(
                emp_row[2]
            )

            dept_key = int(
                dept_row[0]
            )

            # Matching key
            if emp_key == dept_key:

                writer.writerow([
                    emp_row[0],
                    emp_row[1],
                    emp_row[2],
                    dept_row[1]
                ])

                # Move employee stream
                try:
                    emp_row = next(emp_reader)

                except StopIteration:
                    break

            # Employee key is smaller
            elif emp_key < dept_key:

                try:
                    emp_row = next(emp_reader)

                except StopIteration:
                    break

            # Department key is smaller
            else:

                try:
                    dept_row = next(dept_reader)

                except StopIteration:
                    break

In [ ]:
joined_file = "employee_department_join.csv"

streaming_merge_join(
    sorted_employee_file,
    sorted_department_file,
    joined_file
)

print("Streaming Merge Join completed successfully.")

Streaming Merge Join completed successfully.


In [ ]:
with open(
    joined_file,
    "r",
    newline=""
) as file:

    reader = csv.reader(file)

    for i, row in enumerate(reader):

        print(row)

        if i >= 10:
            break

['EmployeeID', 'Name', 'DepartmentID', 'DepartmentName']
['120', 'Employee_120', '1', 'Department_1']
['244', 'Employee_244', '1', 'Department_1']
['301', 'Employee_301', '1', 'Department_1']
['345', 'Employee_345', '1', 'Department_1']
['653', 'Employee_653', '1', 'Department_1']
['974', 'Employee_974', '1', 'Department_1']
['996', 'Employee_996', '1', 'Department_1']
['1077', 'Employee_1077', '1', 'Department_1']
['1175', 'Employee_1175', '1', 'Department_1']
['1298', 'Employee_1298', '1', 'Department_1']


In [ ]:
count = 0

with open(
    joined_file,
    "r",
    newline=""
) as file:

    reader = csv.reader(file)

    # Skip header
    next(reader)

    for row in reader:
        count += 1

print("Total joined records:", count)

Total joined records: 10000


In [ ]:
def cleanup_files(files):

    for file in files:

        if os.path.exists(file):

            os.remove(file)

cleanup_files(employee_chunks)

cleanup_files(department_chunks)

print("Temporary chunk files deleted successfully.")

Temporary chunk files deleted successfully.
